# Kaggle Submission Notebook

This notebook runs the end-to-end pipeline and writes submission.csv to /kaggle/working.
Add the competition dataset and this repository as Kaggle Datasets, then run all cells.


In [ ]:
import os
import shutil
import sys
from pathlib import Path

# ByT5 の選択（base / large）
# ここだけ変更すれば切り替えできます。
BYT5_VARIANT = "base"  # "large" も可
MODEL_DATASET = f"byt5-{BYT5_VARIANT}-model"  # Dataタブ上のDataset名に合わせてください
MODEL_DIR = f"/kaggle/input/{MODEL_DATASET}/byt5-{BYT5_VARIANT}"
os.environ["BYT5_VARIANT"] = BYT5_VARIANT
os.environ["MODEL_DATASET"] = MODEL_DATASET
os.environ["MODEL_DIR"] = MODEL_DIR
print("BYT5_VARIANT:", BYT5_VARIANT)
print("MODEL_DIR:", MODEL_DIR)

def find_repo():
    candidates = [Path.cwd(), Path('/kaggle/working/translate-akkadian')]
    for cand in candidates:
        if (cand / 'src' / 'dp').exists():
            return cand
    input_root = Path('/kaggle/input')
    if input_root.exists():
        for path in input_root.iterdir():
            if (path / 'src' / 'dp').exists():
                return path
            if path.is_dir():
                for sub in path.iterdir():
                    if (sub / 'src' / 'dp').exists():
                        return sub
    return None

repo_env = os.environ.get('REPO_DIR')
repo_src = Path(repo_env) if repo_env else find_repo()
if repo_src is None or not repo_src.exists():
    raise RuntimeError('Repository not found. Add the repo dataset and set REPO_DIR if needed.')

work_dir = Path('/kaggle/working/translate-akkadian')
if str(repo_src).startswith('/kaggle/input'):
    if work_dir.exists():
        shutil.rmtree(work_dir)
    shutil.copytree(repo_src, work_dir)
    repo_dir = work_dir
else:
    repo_dir = repo_src

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir / 'src'))
pythonpath = str(repo_dir / 'src')
os.environ['PYTHONPATH'] = pythonpath + (':' + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')
print('PYTHONPATH:', os.environ['PYTHONPATH'])
print('Repo:', repo_dir)

def find_comp_data():
    env = os.environ.get('COMP_DATA_DIR')
    if env:
        return Path(env)
    input_root = Path('/kaggle/input')
    if input_root.exists():
        for path in input_root.iterdir():
            if (path / 'train.csv').exists() and (path / 'test.csv').exists():
                return path
    return None

data_dir = find_comp_data()
if data_dir is None:
    raise RuntimeError('Competition data not found. Add competition dataset and set COMP_DATA_DIR if needed.')
print('Data dir:', data_dir)

def find_model_dir():
    env = os.environ.get('MODEL_DIR')
    if env:
        p = Path(env)
        if p.exists():
            return p
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None

    def looks_like_model(path: Path) -> bool:
        config = path / 'config.json'
        tokenizer_json = path / 'tokenizer.json'
        spm = path / 'spiece.model'
        if not config.exists():
            return False
        if not (tokenizer_json.exists() or spm.exists()):
            return False
        has_bin = any(path.glob('pytorch_model*.bin')) or (path / 'pytorch_model.bin.index.json').exists()
        has_safetensors = any(path.glob('model*.safetensors')) or (path / 'model.safetensors.index.json').exists()
        return has_bin or has_safetensors

    candidates = []
    for path in input_root.iterdir():
        if not path.is_dir():
            continue
        if looks_like_model(path):
            candidates.append(path)
        for sub in path.iterdir():
            if sub.is_dir() and looks_like_model(sub):
                candidates.append(sub)

    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        print('Model candidates:', candidates)
    return None

model_dir = find_model_dir()
if model_dir:
    os.environ['MODEL_DIR'] = str(model_dir)
    print('Model dir:', model_dir)
else:
    print('Model dir: not found (set MODEL_DIR if offline)')


In [ ]:
import importlib

# Kaggle の提出(推論)では sacrebleu は必須ではありません。
# sacrebleu が無い場合、BLEU/chrF++ などの評価ログだけスキップします。
required = ['pandas', 'sklearn', 'pyarrow', 'torch', 'transformers', 'datasets', 'sentencepiece']
optional = ['sacrebleu']

missing_required = []
for pkg in required:
    try:
        importlib.import_module(pkg)
    except Exception:
        missing_required.append(pkg)

missing_optional = []
for pkg in optional:
    try:
        importlib.import_module(pkg)
    except Exception:
        missing_optional.append(pkg)

if missing_required:
    raise RuntimeError(f"Missing required packages: {missing_required}")
if missing_optional:
    print(f"Optional packages missing: {missing_optional} (metrics will be skipped)")
print('Dependencies OK')


In [ ]:
RUN_TRAIN = True
RUN_OCR = False  # OCR追加データを使う場合は True
SHOW_OCR_STATS = True  # OCR統計ログを表示
INFER_CKPT_DIR = None
OCR_CONFIG = "configs/ocr.yaml"

import subprocess

def run(cmd):
    if cmd.startswith('python '):
        cmd = cmd.replace('python ', f"{sys.executable} ", 1)
    print(cmd)
    subprocess.check_call(cmd, shell=True)

ocr_stats_paths = [
    Path('artifacts/ocr/publications_candidates_stats.json'),
    Path('artifacts/ocr_pairs/summary.json'),
    Path('artifacts/ocr_pairs/mixed_train.stats.json'),
]
train_path = Path('artifacts/aligned/aligned_train.parquet')

if RUN_TRAIN:
    run(f"python -m dp.align_train --config configs/align.yaml --data-dir '{data_dir}' --variant C --drop-flagged")
    if RUN_OCR:
        pub_path = data_dir / 'publications.csv'
        if not pub_path.exists():
            raise RuntimeError(f'publications.csv not found: {pub_path}')
        run(f"python -m dp.ocr_candidates --config '{OCR_CONFIG}' --input '{pub_path}'")
        run(f"python -m dp.ocr_pairs --config '{OCR_CONFIG}'")
        run(f"python -m dp.mix_ocr --config '{OCR_CONFIG}'")
        train_path = Path('artifacts/ocr_pairs/mixed_train.parquet')
    ckpt_dir = Path('artifacts/nmt/byt5_small')
else:
    if INFER_CKPT_DIR:
        ckpt_dir = Path(INFER_CKPT_DIR)
    else:
        model_env = os.environ.get('MODEL_DIR')
        ckpt_dir = Path(model_env) if model_env else None
    if ckpt_dir is None or not ckpt_dir.exists():
        raise RuntimeError('Model dir not found. Set INFER_CKPT_DIR or MODEL_DIR when RUN_TRAIN is False.')
print('RUN_TRAIN:', RUN_TRAIN)
print('RUN_OCR:', RUN_OCR)
print('TRAIN_PATH:', train_path)
print('CKPT_DIR:', ckpt_dir)
if RUN_OCR and SHOW_OCR_STATS:
    import json

    def show_stats(path):
        if not path.exists():
            print(f'[stats] missing: {path}')
            return
        data = json.loads(path.read_text())
        print(f'[stats] {path}')
        print(json.dumps(data, indent=2, ensure_ascii=False))

    for path in ocr_stats_paths:
        show_stats(path)
else:
    print('OCR統計ログはスキップします')


In [ ]:
from pathlib import Path
import json

from dp.utils import load_config

base_config = Path('configs/nmt_byt5_small.yaml')
runtime_config = Path('configs/nmt_byt5_small.runtime.json')

overrides = {
    'model_name_or_path': None,
    'max_source_length': None,
    'max_target_length': None,
    'val_ratio': None,
    'seed': None,
    'learning_rate': None,
    'num_train_epochs': None,
    'per_device_train_batch_size': None,
    'per_device_eval_batch_size': None,
    'gradient_accumulation_steps': None,
    'weight_decay': None,
    'max_grad_norm': None,
    'warmup_ratio': None,
    'lr_scheduler_type': None,
    'step_decay_steps': None,
    'step_decay_gamma': None,
    'logging_steps': None,
    'eval_steps': None,
    'save_steps': None,
    'save_total_limit': None,
    'predict_with_generate': None,
    'generation_max_length': None,
    'generation_max_new_tokens': None,
    'num_beams': None,
    'length_penalty': None,
    'early_stopping': None,
    'no_repeat_ngram_size': None,
    'repetition_penalty': None,
    'fp16': None,
    'gradient_checkpointing': None,
    'dropout_rate': None,
    'attention_dropout_rate': None,
}

cfg = load_config(str(base_config))
override_items = {k: v for k, v in overrides.items() if v is not None}
if override_items:
    cfg.update(override_items)
    runtime_config.write_text(json.dumps(cfg, ensure_ascii=False, indent=2))
    config_path = runtime_config
else:
    config_path = base_config
print('Config path:', config_path)
print('Overrides:', sorted(override_items.keys()))


In [ ]:
if RUN_TRAIN:
    model_dir = os.environ.get('MODEL_DIR')
    model_arg = f" --model-name-or-path '{model_dir}'" if model_dir else ''
    run(
        f"python -m dp.train_nmt --config '{config_path}' "
        f"--train '{train_path}' --variant C --drop-flagged "
        f"--out '{ckpt_dir}'" + model_arg
    )
else:
    print('Skip training: RUN_TRAIN is False')


In [ ]:
run(
    f"python -m dp.infer_nmt --config '{config_path}' "
    f"--ckpt '{ckpt_dir}' "
    f"--test '{data_dir / 'test.csv'}' --norm-variant C "
    "--out predictions.csv"
)


In [ ]:
submission_out = '/kaggle/working/submission.csv'
run(
    "python -m dp.submit --pred predictions.csv "
    f"--out '{submission_out}' --data-dir '{data_dir}'"
)


In [ ]:
from pathlib import Path
import shutil

submission_target = Path('/kaggle/working/submission.csv')
candidates = [submission_target, Path('submission.csv'), Path('artifacts') / 'submission.csv']
if not submission_target.exists():
    for cand in candidates:
        if cand.exists():
            shutil.copy2(cand, submission_target)
            break

if not submission_target.exists():
    raise RuntimeError('submission.csv not found in expected locations')

print('submission.csv:', submission_target, 'size', submission_target.stat().st_size)


In [ ]:
run(f"python -m dp.validate --submission '{submission_out}' --data-dir '{data_dir}'")


submission.csv is saved under /kaggle/working. Use it as the notebook output file for submission.
